<a href="https://colab.research.google.com/github/GustavoKozan/IC-UFPR-Agents/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CÉLULA 1: INSTALAÇÃO E CONFIGURAÇÃO DO AMBIENTE DE BENCHMARK (CORRIGIDA)
# ==============================================================================
import os
import sys

print("Instalando dependências do benchmark...")

# 1. Instalação das dependências analíticas padrão
!pip install -q requests tqdm pandas openpyxl tabulate radon

# 2. Descarrega e corrige a sintaxe legada do setup.py do HumanEval para o pip moderno
!rm -rf human-eval
!git clone -q https://github.com/openai/human-eval.git
!sed -i 's/human_eval.evaluate_functional_correctness/human_eval.evaluate_functional_correctness:main/g' human-eval/setup.py

# 3. Instalação do pacote corrigido no ambiente
!pip install -q ./human-eval

# 4. Verificação de integridade das importações
try:
    from human_eval.data import read_problems
    from human_eval.execution import check_correctness
    print("\n✓ Framework HumanEval instalado e validado com sucesso!")
    print("✓ Ambiente 100% operacional para executar as Células 2 e 3.")
except ImportError as err:
    raise RuntimeError(f"Falha ao validar importação do human-eval: {err}")

Instalando dependências do benchmark...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.5 MB/s eta 0:00:00

✓ Framework HumanEval instalado e validado com sucesso!
✓ Ambiente 100% operacional para executar as Células 2 e 3.


In [ ]:
import os
import site
import urllib.request

# Encontra o diretório site-packages atual dinamicamente independente da versão do Python
site_packages = site.getsitepackages()[0]
target_dir = os.path.join(site_packages, "human_eval", "data")
os.makedirs(target_dir, exist_ok=True)

target_file = os.path.join(target_dir, "HumanEval.jsonl.gz")
url = "https://github.com/openai/human-eval/raw/master/data/HumanEval.jsonl.gz"

print(f"Baixando HumanEval.jsonl.gz para:\n{target_file}")
urllib.request.urlretrieve(url, target_file)
print("✓ Download concluído com sucesso!")

Baixando HumanEval.jsonl.gz para:
/usr/local/lib/python3.13/dist-packages/human_eval/data/HumanEval.jsonl.gz
✓ Download concluído com sucesso!


In [ ]:
# ==============================================================================
# CÉLULA 2: MOTOR UNIVERSAL DE INFERÊNCIA MULTI-PROVEDOR
# ==============================================================================
import json
import os
from pathlib import Path
import re
import textwrap
import time
from google.colab import userdata
from human_eval.data import read_problems
import requests
from tqdm import tqdm

# ------------------------------------------------------------------------------
# 1. ESCOLHA DO MODELO (Altere apenas esta linha para testar qualquer modelo)
# ------------------------------------------------------------------------------
# Exemplos Google AI Studio:
#   "gemini-3.5-flash-lite"
#   "gemini-2.5-flash"
#   "gemini-2.0-flash"
#   "gemini-2.0-flash-lite"
#   "gemini-1.5-pro"
#
# Exemplos NVIDIA NIM:
#   "moonshotai/kimi-k3"
#   "meta/llama-3.3-70b-instruct"
#   "deepseek-ai/deepseek-r1"
#   "qwen/qwen2.5-72b-instruct"
#
# Exemplos Groq:
#   "openai/gpt-oss-120b"
#   "qwen/qwen3.8-27b"

MODEL_NAME = "gemini-3.5-flash-lite"

# Seletor de Provedor: "auto" detecta sozinho pelo nome do modelo,
# ou force manualmente: "google", "nvidia", "groq", "openrouter"
PROVIDER = "auto"

# ------------------------------------------------------------------------------
# 2. ROTEAMENTO DINÂMICO DE PROVEDORES E CREDENCIAIS
# ------------------------------------------------------------------------------
CONFIGS = {
    "google": {
        "url": (
            "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
        ),
        "keys": ["GEMINI_API_KEY", "GOOGLE_API_KEY"],
        "delay": 4.5,  # Respeita o teto de 15 RPM da camada gratuita
    },
    "nvidia": {
        "url": "https://integrate.api.nvidia.com/v1/chat/completions",
        "keys": ["NVIDIA_API_KEY"],
        "delay": 5.5,  # Evita o limite de 10-12 RPM da NVIDIA
    },
    "groq": {
        "url": "https://api.groq.com/openai/v1/chat/completions",
        "keys": ["GROQ_API_KEY"],
        "delay": 1.5,
    },
    "openrouter": {
        "url": "https://openrouter.ai/api/v1/chat/completions",
        "keys": ["OPENROUTER_API_KEY"],
        "delay": 4.5,
    },
}


# Detecção automática do provedor se estiver em "auto"
def resolver_provedor(modelo: str, prov_sel: str) -> str:
  if prov_sel != "auto":
    return prov_sel
  m = modelo.lower()
  if m.startswith("gemini"):
    return "google"
  elif (
      m.startswith("moonshotai/")
      or m.startswith("nvidia/")
      or m.startswith("meta/")
      or m.startswith("deepseek-ai/")
      or m.startswith("mistralai/")
  ):
    return "nvidia"
  elif "groq" in m or "openai/gpt-oss" in m:
    return "groq"
  elif userdata.get("NVIDIA_API_KEY"):
    return "nvidia"
  return "google"


provedor_ativo = resolver_provedor(MODEL_NAME, PROVIDER)
cfg = CONFIGS[provedor_ativo]

# Busca a chave de API nos Secrets do Colab
api_key = None
for k_name in cfg["keys"]:
  val = userdata.get(k_name)
  if val:
    api_key = str(val).strip().replace('"', "").replace("'", "")
    break

if not api_key:
  raise ValueError(
      f"Chave para o provedor '{provedor_ativo.upper()}' não encontrada nos"
      f" Secrets!\nCadastre uma das chaves: {cfg['keys']} com a opção 'Notebook"
      " access' ativada."
  )

INVOKE_URL = cfg["url"]
DELAY_CALLS = cfg["delay"]
headers = {
    "Authorization": f"Bearer {api_key}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

# ------------------------------------------------------------------------------
# 3. ISOLAMENTO DINÂMICO DE ARQUIVOS (Evita sobrescrever outros modelos)
# ------------------------------------------------------------------------------
MODEL_SLUG = re.sub(r"[^a-zA-Z0-9_-]", "_", MODEL_NAME)
output_samples = Path(f"samples_{MODEL_SLUG}.jsonl")
output_metrics = Path(f"metrics_cache_{MODEL_SLUG}.json")

# ------------------------------------------------------------------------------
# 4. TESTE RÁPIDO DE CONECTIVIDADE
# ------------------------------------------------------------------------------
print(
    f"Validando conexão com [{MODEL_NAME}] via"
    f" {provedor_ativo.upper()} (Google AI Studio)..."
    if provedor_ativo == "google"
    else f"Validando conexão com [{MODEL_NAME}] via {provedor_ativo.upper()}..."
)

try:
  teste_resp = requests.post(
      INVOKE_URL,
      headers=headers,
      json={
          "model": MODEL_NAME,
          "messages": [{"role": "user", "content": "print('ok')"}],
          "max_tokens": 15,
          "temperature": 0.1,
          "stream": False,
      },
      timeout=30.0,
  )
  if teste_resp.status_code != 200:
    raise RuntimeError(f"HTTP {teste_resp.status_code}: {teste_resp.text}")
  print("✓ Conexão e autenticação estabelecidas com sucesso!\n")
except Exception as e:
  raise RuntimeError(
      f"Falha de conexão com a API de {provedor_ativo.upper()}: {e}"
  )

# ------------------------------------------------------------------------------
# 5. PROMPT DE SISTEMA E SANITIZAÇÃO ROBUSTA (Idênticos aos anteriores)
# ------------------------------------------------------------------------------
SYSTEM_PROMPT = (
    "Você é um motor de geração de código Python especializado.\n"
    "REGRAS:\n"
    "1. Complete a função solicitada sem nenhum texto conversacional,"
    " explicação ou saudação.\n"
    "2. Inicie sua resposta estritamente com o bloco de código ```python e feche"
    " com ```.\n"
    "3. NÃO altere o nome da função nem a sua assinatura."
)


def sanitize_completion(raw_text: str, original_prompt: str) -> str:
  if not raw_text or not raw_text.strip():
    return ""

  # Remove tags de pensamento de modelos analíticos
  cleaned = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
  if not cleaned:
    cleaned = raw_text

  # Captura o último bloco de código (solução definitiva)
  matches = list(
      re.finditer(r"```(?:python)?\s*\n?(.*?)(?:```|$)", cleaned, flags=re.DOTALL)
  )
  clean = matches[-1].group(1).strip() if matches else cleaned.strip()

  # Remove cabeçalho e docstring repetidos se o modelo tiver duplicado
  func_match = re.search(r"def\s+([a-zA-Z0-9_]+)\s*\(", original_prompt)
  if func_match:
    func_name = func_match.group(1)
    lines = clean.splitlines()
    def_line_idx = -1
    for idx, line in enumerate(lines):
      if re.match(rf"^\s*def\s+{func_name}\s*\(", line):
        def_line_idx = idx
        break

    if def_line_idx != -1:
      lines = lines[def_line_idx + 1 :]
      while lines and not lines[0].strip():
        lines = lines[1:]
      if lines and (
          lines[0].strip().startswith('"""')
          or lines[0].strip().startswith("'''")
      ):
        q = lines[0].strip()[:3]
        lines = lines[1:]
        while lines:
          if q in lines[0]:
            lines = lines[1:]
            break
          lines = lines[1:]
      clean = "\n".join(lines)

  dedented = textwrap.dedent(clean).strip("\n")
  return textwrap.indent(dedented, "    ") if dedented else ""


# ------------------------------------------------------------------------------
# 6. CHAMADA SEGURA COM RECUPERAÇÃO DE REASONING E RATE LIMIT
# ------------------------------------------------------------------------------
def call_api_safe(messages: list, max_tentativas=8):
  payload = {
      "model": MODEL_NAME,
      "messages": messages,
      "temperature": 0.1,
      "max_tokens": 8192,  # Teto amplo: impede truncamento de raciocínio
      "stream": False,
  }

  for tentativa in range(max_tentativas):
    try:
      start = time.perf_counter()
      response = requests.post(
          INVOKE_URL, headers=headers, json=payload, timeout=90.0
      )
      latency = time.perf_counter() - start

      # Tratamento dinâmico de 429 para Google AI Studio e NVIDIA
      if response.status_code == 429:
        retry_h = response.headers.get("Retry-After")
        wait_time = (
            float(retry_h) if retry_h else max(15.0, 15.0 * (1.3**tentativa))
        )
        print(
            f"\n[Rate Limit 429] Aguardando {wait_time:.1f}s exigidos pelo"
            f" servidor ({provedor_ativo})..."
        )
        time.sleep(wait_time)
        continue

      if response.status_code != 200:
        time.sleep(4.0 * (tentativa + 1))
        continue

      data = response.json()
      choice = data["choices"][0]
      msg = choice.get("message", {}).get("content", "") or ""

      # Fallback seguro caso raciocínio venha separado em modelos analíticos
      if not msg.strip() and choice.get("message", {}).get("reasoning_content"):
        msg = choice.get("message", {}).get("reasoning_content", "")

      usage = data.get("usage", {})
      return (
          msg,
          latency,
          usage.get("prompt_tokens", 0),
          usage.get("completion_tokens", 0),
          usage.get("total_tokens", 0),
      )
    except Exception as e:
      if tentativa == max_tentativas - 1:
        raise e
      time.sleep(3.5 * (tentativa + 1))

  return "", 0.0, 0, 0, 0


def run_agent_task(prompt_str: str):
  msgs = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {
          "role": "user",
          "content": f"Complete a seguinte função em Python:\n\n{prompt_str}",
      },
  ]
  raw, lat, p, c, t = call_api_safe(msgs)
  comp = sanitize_completion(raw, prompt_str)
  return comp, {
      "latency_sec": lat,
      "prompt_tokens": p,
      "completion_tokens": c,
      "total_tokens": t,
      "tentativas": 1,
  }


# ------------------------------------------------------------------------------
# 7. EXECUÇÃO COM CHECKPOINTING E AUTOCURA DE RESPOSTAS VAZIAS
# ------------------------------------------------------------------------------
amostras_salvas = {}
metrics_log = {}

if output_samples.exists():
  with open(output_samples, "r", encoding="utf-8") as f:
    for line in f:
      if line.strip():
        try:
          d = json.loads(line)
          amostras_salvas[d["task_id"]] = d["completion"]
        except Exception:
          pass

if output_metrics.exists():
  try:
    with open(output_metrics, "r", encoding="utf-8") as f:
      metrics_log = json.load(f)
  except Exception:
    metrics_log = {}

all_problems = read_problems()
total_tarefas = len(all_problems)

# Validação: só considera concluída se tiver código não vazio
tarefas_concluidas = {
    t_id
    for t_id, comp in amostras_salvas.items()
    if comp and str(comp).strip()
}
tarefas_pendentes = {
    k: v for k, v in all_problems.items() if k not in tarefas_concluidas
}

print(f"Benchmark para '{MODEL_NAME}' [{provedor_ativo.upper()}]:")
print(f"• Total de tarefas: {total_tarefas}")
print(f"• Tarefas concluídas com sucesso: {len(tarefas_concluidas)}")
print(f"• Tarefas pendentes: {len(tarefas_pendentes)}\n")

for task_id, problem in tqdm(
    tarefas_pendentes.items(),
    total=len(tarefas_pendentes),
    desc=f"Processando {MODEL_SLUG[:15]}",
):
  try:
    comp, met = run_agent_task(problem["prompt"])
    metrics_log[task_id] = met
    amostras_salvas[task_id] = comp
  except Exception as exc:
    print(f"\n[Erro na tarefa {task_id}]: {exc}")
    comp = ""
    amostras_salvas[task_id] = ""
    metrics_log[task_id] = {
        "latency_sec": 0.0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "tentativas": 0,
        "erro": str(exc),
    }

  # Gravação imediata de cada tarefa em disco
  with open(output_samples, "w", encoding="utf-8") as f:
    for tid, c_text in amostras_salvas.items():
      f.write(json.dumps({"task_id": tid, "completion": c_text}) + "\n")

  with open(output_metrics, "w", encoding="utf-8") as f:
    json.dump(metrics_log, f)

  time.sleep(DELAY_CALLS)

print(
    f"\n✓ Execução concluída! Todas as tarefas salvas em '{output_samples.name}'."
)

ModuleNotFoundError: No module named 'human_eval'

In [ ]:
# ==============================================================================
# CÉLULA 3: AVALIAÇÃO DE PASS@1, MÉTRICAS RADON E EXPORTAÇÃO
# ==============================================================================
import json
from pathlib import Path
import re
import sys
from human_eval.data import read_problems
from human_eval.execution import check_correctness
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import pandas as pd
from radon.complexity import cc_visit
from radon.metrics import h_visit, mi_visit
from radon.raw import analyze
from tabulate import tabulate
from tqdm import tqdm

try:
  TARGET_MODEL = MODEL_NAME
except NameError:
  TARGET_MODEL = "gemini-2.5-flash"

TARGET_SLUG = re.sub(r"[^a-zA-Z0-9_-]", "_", TARGET_MODEL)
samples_path = Path(f"samples_{TARGET_SLUG}.jsonl")
metrics_path = Path(f"metrics_cache_{TARGET_SLUG}.json")

if not samples_path.exists():
  raise FileNotFoundError(
      f"Arquivo '{samples_path}' não encontrado! Execute a Célula 2 primeiro."
  )

perf_metrics = {}
if metrics_path.exists():
  with open(metrics_path, "r", encoding="utf-8") as f:
    perf_metrics = json.load(f)

samples_dict = {}
with open(samples_path, "r", encoding="utf-8") as f:
  for line in f:
    if line.strip():
      item = json.loads(line)
      samples_dict[item["task_id"]] = item["completion"]

all_problems = read_problems()
print(
    f"Avaliando {len(samples_dict)} tarefas resolvidas por '{TARGET_MODEL}'..."
)

eval_results = []
for t_id, comp in tqdm(
    samples_dict.items(), total=len(samples_dict), desc="Avaliando Pass@1"
):
  if t_id in all_problems:
    res = check_correctness(all_problems[t_id], comp, timeout=3.0)
    eval_results.append(res)

df_eval = pd.DataFrame(eval_results)
df_eval["completion"] = df_eval["task_id"].map(samples_dict)

radon_records = []
for _, row in df_eval.iterrows():
  t_id = row["task_id"]
  comp = row["completion"]
  full_code = all_problems[t_id]["prompt"] + (comp if comp else "")

  cc_score = 1
  mi_score = 0.0
  h_volume = 0.0
  h_difficulty = 0.0
  sloc = 0

  if comp and str(comp).strip():
    try:
      cc_blocks = cc_visit(full_code)
      cc_score = max([b.complexity for b in cc_blocks]) if cc_blocks else 1
      mi_score = mi_visit(full_code, multi=True)
      h_metrics = h_visit(full_code)
      h_volume = h_metrics.total.volume
      h_difficulty = h_metrics.total.difficulty
      raw_metrics = analyze(full_code)
      sloc = raw_metrics.sloc
    except Exception:
      pass

  perf = perf_metrics.get(t_id, {})
  raw_res = (
      str(row.get("result", "")).replace("\n", " ").replace("\r", " ").strip()
  )

  radon_records.append({
      "ID_Tarefa": t_id,
      "Status": "Aprovado" if bool(row["passed"]) else "Falhou",
      "Resultado_Execucao": raw_res[:50],
      "Latencia_Segundos": round(float(perf.get("latency_sec", 0.0)), 2),
      "Tokens_Prompt": int(perf.get("prompt_tokens", 0)),
      "Tokens_Resposta": int(perf.get("completion_tokens", 0)),
      "Total_Tokens": int(perf.get("total_tokens", 0)),
      "Complexidade_Ciclomatica": int(cc_score),
      "Indice_Manutenibilidade": round(float(mi_score), 2),
      "Volume_Halstead": round(float(h_volume), 2),
      "Dificuldade_Halstead": round(float(h_difficulty), 2),
      "Linhas_Codigo_SLOC": int(sloc),
  })

df_final = pd.DataFrame(radon_records)
df_final["_ordem"] = df_final["ID_Tarefa"].apply(
    lambda x: int(x.split("/")[-1]) if "/" in str(x) else 0
)
df_final = (
    df_final.sort_values(by="_ordem")
    .drop(columns=["_ordem"])
    .reset_index(drop=True)
)

total_tasks = len(df_final)
total_passed = int((df_final["Status"] == "Aprovado").sum())
pass_rate = (total_passed / total_tasks * 100) if total_tasks > 0 else 0.0
error_rate = 100.0 - pass_rate

resumo_geral = [
    {
        "Métrica": "Total de Tarefas Avaliadas",
        "Valor": total_tasks,
        "Unidade": "tarefas",
    },
    {
        "Métrica": "Soluções Aprovadas (Pass@1)",
        "Valor": f"{pass_rate:.2f}%",
        "Unidade": f"{total_passed}/{total_tasks}",
    },
    {
        "Métrica": "Taxa de Falha",
        "Valor": f"{error_rate:.2f}%",
        "Unidade": f"{total_tasks - total_passed}/{total_tasks}",
    },
    {
        "Métrica": "Latência Média por Requisição",
        "Valor": round(df_final["Latencia_Segundos"].mean(), 2),
        "Unidade": "segundos",
    },
    {
        "Métrica": "Total de Tokens Processados",
        "Valor": int(df_final["Total_Tokens"].sum()),
        "Unidade": "tokens",
    },
    {
        "Métrica": "Tokens Médios de Resposta",
        "Valor": round(df_final["Tokens_Resposta"].mean(), 1),
        "Unidade": "tokens/tarefa",
    },
    {
        "Métrica": "Complexidade Ciclomática Média",
        "Valor": round(df_final["Complexidade_Ciclomatica"].mean(), 2),
        "Unidade": "escala CC",
    },
    {
        "Métrica": "Índice de Manutenibilidade Médio",
        "Valor": round(df_final["Indice_Manutenibilidade"].mean(), 2),
        "Unidade": "escala 0-100",
    },
    {
        "Métrica": "Volume Halstead Médio",
        "Valor": round(df_final["Volume_Halstead"].mean(), 2),
        "Unidade": "escala Halstead",
    },
    {
        "Métrica": "Dificuldade Halstead Média",
        "Valor": round(df_final["Dificuldade_Halstead"].mean(), 2),
        "Unidade": "escala Halstead",
    },
    {
        "Métrica": "SLOC Médio (Linhas de Código)",
        "Valor": round(df_final["Linhas_Codigo_SLOC"].mean(), 1),
        "Unidade": "linhas",
    },
]
df_resumo = pd.DataFrame(resumo_geral)

csv_filename = f"relatorio_{TARGET_SLUG}_analitico.csv"
excel_filename = f"relatorio_{TARGET_SLUG}_completo.xlsx"
df_final.to_csv(csv_filename, sep=";", index=False, encoding="utf-8-sig")

wb = Workbook()
ws_res = wb.active
ws_res.title = "Resumo Executivo"
for r in dataframe_to_rows(df_resumo, index=False, header=True):
  ws_res.append(r)
ws_tar = wb.create_sheet(title="Métricas por Tarefa")
for r in dataframe_to_rows(df_final, index=False, header=True):
  ws_tar.append(r)
wb.save(excel_filename)

print("\n" + "=" * 80)
print(f"           RESUMO EXECUTIVO: {TARGET_MODEL}")
print("=" * 80)
print(
    tabulate(df_resumo, headers="keys", tablefmt="fancy_grid", showindex=False)
)
print(f"\n✓ Relatórios salvos: '{csv_filename}' e '{excel_filename}'.")

Avaliando 164 tarefas resolvidas por 'gemini-3.5-flash-lite'...


Avaliando Pass@1: 100%|██████████| 164/164 [00:07<00:00, 22.71it/s]



           RESUMO EXECUTIVO: gemini-3.5-flash-lite
╒══════════════════════════════════╤═════════╤═════════════════╕
│ Métrica                          │ Valor   │ Unidade         │
╞══════════════════════════════════╪═════════╪═════════════════╡
│ Total de Tarefas Avaliadas       │ 164     │ tarefas         │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Soluções Aprovadas (Pass@1)      │ 88.41%  │ 145/164         │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Taxa de Falha                    │ 11.59%  │ 19/164          │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Latência Média por Requisição    │ 1.24    │ segundos        │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Total de Tokens Processados      │ 60354   │ tokens          │
├──────────────────────────────────┼─────────┼─────────────────┤
│ Tokens Médios de Resposta        │ 138.7   │ tokens/tarefa   │
├──────────────────────────────────┼──

In [ ]:
# Teste isolado utilizando as variáveis globais já carregadas na Célula 2
# Certifique-se de ter rodado a Célula 2 para carregar o modelo e a chave

api_key_test = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY") or userdata.get("OPENROUTER_API_KEY")
api_key_test = str(api_key_test).strip().replace('"', "")

# Testa no endpoint do Google AI Studio
url_google = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

headers_test = {
    "Authorization": f"Bearer {api_key_test}",
    "Content-Type": "application/json"
}

payload_test = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "print('ok')"}],
    "max_tokens": 10
}

r = requests.post(url_google, headers=headers_test, json=payload_test, timeout=15.0)
print(f"Status HTTP: {r.status_code}")
if r.status_code == 200:
    print(f"✓ Modelo '{MODEL_NAME}' aceito com sucesso!")
else:
    print(f"Resposta do servidor:\n{r.text}")

Status HTTP: 200
✓ Modelo 'gemini-3.5-flash-lite' aceito com sucesso!
